In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (CAPTP)

This notebook curates the **CAPTP** dataset from a collection of FASTA files. The pipeline parses all sequences, extracts binary labels directly from the FASTA record identifiers, performs duplicate consistency checks, and exports a standardized dataset plus metadata for downstream analysis.

- **Toxic effect / endpoint:** toxic
- **Source:** CAPTP
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset

The pipeline performs the following steps:
- **Loads all FASTA files** found under the CAPTP input directory and concatenates them into a single table.
- **Extracts labels from FASTA record IDs** using the convention:
  - `id` is split by `"|"`,
  - the second field (`id.split("|")[1]`) is interpreted as an integer label.
- **Keeps a standardized schema**:
  - `sequence`
  - `label`
- **Checks duplicates by sequence**:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description file and appends QC statistics.
- **Exports outputs**:
  - `processed_toxic_dataset.csv` (deduplicated curated dataset),
  - `metadata.json`.
- **Outputs and assumptions**
    - Label extraction assumes a strict header format containing at least two `"|"`-separated fields.
    - No sequence modification handling is performed (`modified_sequences_included = False`).

In [2]:
name_source = "CAPTP"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
input_dir = Path(PATH_INPUT) / name_source
dfs = []

for file in input_dir.iterdir():
    df = read_fasta_doc(file)
    dfs.append(df)
df_CATP = pd.concat(dfs, ignore_index=True)

In [4]:
df_CATP = (
    df_CATP
    .assign(
        label=lambda d: d["id"].str.split("|").str[1].astype(int)
    )
    [["sequence", "label"]]
)
df_CATP.shape

(8095, 2)

- Checking duplicates

In [5]:
df_CATP["sequence"].unique().shape

(7513,)

In [6]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_CATP, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [7]:
df_full.shape

(7513, 2)

In [8]:
df_errors.shape

(0, 1)

- Working with metada

In [9]:
df_metada = read_metadata_multiple("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada) 

In [10]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_CATP)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2024,
 'last update date': datetime.datetime(2025, 9, 24, 0, 0),
 'download date': Timestamp('2025-10-17 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Previously published model dataset',
 'repository or server': 'https://github.com/jiaoshihu/CAPTP/tree/main/data',
 'publication': 'https://academic.oup.com/bioinformatics/article/40/5/btae297/7663469',
 'number_of_raw_sequences': 8095,
 'number_of_sequences_retained': 7513,
 'number_of_positive_sequences': 2138,
 'number_of_negative_sequences': 5375,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [11]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [12]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)